# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/ecg/ecg.parquet'
df_ecg = read_parquet(pfile)
print(df_ecg.shape)
df_ecg.head()

In [ ]:
df_ecg = pd.concat([df_ecg.iloc[:, :7], df_ecg.iloc[:, 11:-3]], axis=1)

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/ecg/ecg_qc.parquet'
df_ecg_qc = read_parquet(pfile)
print(df_ecg_qc.shape)
df_ecg_qc.head()

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_ecg)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Imputation

In [ ]:
ecg_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'conclusion']
ecg_metadata_col = [col for col in ecg_metadata_col if col in df_ecg.columns]
df_ecg_metadata = df_ecg[ecg_metadata_col]
df_ecg_features = df_ecg.drop(columns=ecg_metadata_col)

In [ ]:
missing_ratio_per_feature = df_ecg_features.isnull().sum() / len(df_ecg_features)  # 计算每列的缺失值比例
feature_threshold = 0.5  # 设定特征缺失率阈值

# 剔除缺失率超过阈值的特征
df_ecg_filtered = df_ecg_features.loc[:, missing_ratio_per_feature <= feature_threshold]
print(df_ecg_filtered.shape)

In [ ]:
means = df_ecg_filtered.mean()
df_ecg_imputed = df_ecg_filtered.fillna(means)

# Save

In [ ]:
df_ecg_metadata.reset_index(drop=True, inplace=True)
df_ecg_imputed.reset_index(drop=True, inplace=True)
df_ecg = pd.concat([df_ecg_metadata, df_ecg_imputed], axis=1).dropna(axis=1)
pfile = 'pheno_data/202512/015-ecg_imputed.parquet'
df_ecg.to_parquet(pfile, index=False)

In [ ]:
pfile = 'pheno_data/202512/015-ecg_imputed.parquet'
df_ecg = pd.read_parquet(pfile)
print(df_ecg.shape)
df_ecg.head()